In [4]:
import pandas as pd
import numpy as np
from scipy.stats import norm


In [6]:
data=pd.read_csv(r"C:\Users\Pravalika.b\OneDrive\Documents\a b test.csv")
data.head()


,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [8]:
data.columns

Index(['user_id', 'timestamp', 'group', 'landing_page', 'converted'], dtype='object')

In [10]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64


In [12]:
print("\ngroup counts")
print(data['group'].value_counts())


group counts
group
treatment    147276
control      147202
Name: count, dtype: int64


In [13]:
print("\nconversion rates")
print(pd.crosstab(data['group'],data['converted']))


conversion rates
converted       0      1
group                   
control    129479  17723
treatment  129762  17514


In [15]:
summary=data.groupby("group")["converted"].agg(users="count",conersions="sum",conversion_rate="mean").reset_index()
summary["conversion_rate"]=summary["conversion_rate"]*100
print("\nconversion rate comparision")
print(summary)


conversion rate comparision
       group   users  conersions  conversion_rate
0    control  147202       17723        12.039918
1  treatment  147276       17514        11.891958


In [16]:
control=data[data["group"].str.lower()=="control"]["converted"]
treatment=data[data["group"].str.lower()=="treatment"]["converted"]
n_control=len(control)
n_treatment=len(treatment)
x_control=control.sum()
x_treatment=treatment.sum()
p_control=x_control/n_control
p_treatment=x_treatment/n_treatment

In [18]:
absolute_difference=p_treatment-p_control
relative_lift=(p_treatment-p_control)/p_control
print("\neffect size")
print('control conversion rate',round(p_control*100,2),"%")
print('treatment conversion rate',round(p_treatment*100,2),"%")
print('absolute difference',round(absolute_difference*100,2),"percentage points")
print('relative lift',round(relative_lift*100,2),"%")


effect size
control conversion rate 12.04 %
treatment conversion rate 11.89 %
absolute difference -0.15 percentage points
relative lift -1.23 %


In [20]:
pooled_p=(x_control+x_treatment)/(n_control+n_treatment)
standard_error=np.sqrt(
    pooled_p*(1-pooled_p)*
    (1/n_control+1/n_treatment)
    )
z_score=absolute_difference/standard_error
p_value=2*(1-norm.cdf(abs(z_score)))
print("\n A/B test result:")
print("z_score:", round(z_score,4))
print("p_value:",round(p_value,4))


 A/B test result:
z_score: -1.2369
p_value: 0.2161


In [21]:
standard_error_ci=np.sqrt(
    (p_control*(1-p_control)/n_control)+
    (p_treatment*(1-p_treatment)/n_treatment)
)
z_critical=1.96
lower=absolute_difference-z_critical*standard_error_ci
upper=absolute_difference+z_critical*standard_error_ci
print("\n95% confidence interval:")
print("lower:",round(lower*100,2),"percent points")
print("upper:",round(upper*100,2),"percent points")


95% confidence interval:
lower: -0.38 percent points
upper: 0.09 percent points


In [23]:
alpha=0.05
if p_value<alpha:
    significance="statistically significant"
else:
    significance="not statistically significant"
print("\n statistical significance:",significance)


 statistical significance: not statistically significant


In [24]:
if p_value<alpha and p_treatment>p_control:
    recommendation="choose treatment. the treatment has a statistically significant improvement."
elif p_value<alpha and p_treatment<p_control:
    recommendation="keep control. the treatment performs significantly worse."
else:
    recommendation="keep control for now. the observed difference is not statistically significant."
print("\n recommendation")
print(recommendation)


 recommendation
keep control for now. the observed difference is not statistically significant.


In [26]:
result=pd.DataFrame({
    "metric":[
        "users",
        "conversions",
        "conversion rate",
        "absolute difference",
        "relative lift",
        "z_score",
        "p_value",
        "95% ci lower",
        "95% ci upper",
        "statistical significance"
    ],
    "control":[
        n_control,
        x_control,
        f"{p_control*100:.2f}%",
        "_",
        "_",
        "_",
        "_",
        "_",
        "_",
        "_"
    ],
"treatment":[
n_treatment,
x_treatment,
f"{p_treatment*100:.2f}%",
f"{absolute_difference*100:.2f}pp",
f"{relative_lift*100:.2f}%",
f"{z_score:.4f}",
f"{p_value:.4f}",
f"{lower*100:.2f}pp",
f"{upper*100:.2f}pp",
significance
]
})
print("\n final a/b test comparision table:")
print(result.to_string(index=False))



 final a/b test comparision table:
                  metric control                     treatment
                   users  147202                        147276
             conversions   17723                         17514
         conversion rate  12.04%                        11.89%
     absolute difference       _                       -0.15pp
           relative lift       _                        -1.23%
                 z_score       _                       -1.2369
                 p_value       _                        0.2161
            95% ci lower       _                       -0.38pp
            95% ci upper       _                        0.09pp
statistical significance       _ not statistically significant
